In [1]:
from openai import OpenAI
from openai import AzureOpenAI

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
import os

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery



In [2]:

load_dotenv(override=True) # take environment variables from .env.

endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
credential = AzureKeyCredential(os.environ["AZURE_SEARCH_QUERY_KEY"]) 
index_name = os.environ["AZURE_SEARCH_INDEX"]


# Set the API key and endpoint
api_key = os.getenv('AZURE_OPENAI_API_KEY')
api_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')  # e.g., "https://<your-resource-name>.openai.azure.com/"
api_type = 'azure'
api_version = '2023-05-15'  # Use the appropriate API version

# Define the deployment name
deployment_name_chat = 'gpt-4o-global'
deployment_name_embeddings = 'text-embedding-ada-002'


In [3]:
import re 
def get_page_number(chunk_id : str): 
    page_re = r'_pages_(\d+)$'

    match = re.search(page_re, chunk_id)
    if match:
        page_number = match.group(1)
        return page_number

In [4]:
# Set query parameters for grounding the conversation on your search index
search_type="text"
use_semantic_reranker=True
sources_to_include=5

In [5]:
client = AzureOpenAI(
    azure_endpoint=api_endpoint,
    api_key=api_key,
    api_version=api_version,
)

In [6]:
search_client = SearchClient(
    endpoint, 
    index_name, 
    credential=credential)



In [9]:
 # This prompt provides instructions to the model
 GROUNDED_PROMPT="""
 You are a friendly assistant that answer questions from company's HR policy and information documents.
 Answer the query using only the sources provided below in a friendly and concise bulleted manner.
 Answer ONLY with the facts listed in the list of sources below.
 If there isn't enough information below, say you don't know.
 Do not generate answers that don't use the sources below.
 always provide the source of the information as a citation of source number in the list of sources.
 Sources:\n{sources}

 Semantic Answer:\n{semantic_response}
 """

In [10]:

 # Query is the question being asked. It's sent to the search engine and the LLM.
# user_query="what the role of a project manager?"

user_query = input("Enter your question: ")

In [11]:
#extract user search intent
SEARCH_INTENT_PROMPT = """
You are asked to read the user query, understand it, and then turn it into a search query.
The search query should be sent to the search engine to find the relevant information.
the search query should be similar to what a user would type into a search engine, and similar in meaning to the provided user query.
the user query is: '{user_query}'
return only the search query text. """

print(SEARCH_INTENT_PROMPT.format(user_query=user_query))


response = client.chat.completions.create(
    model=deployment_name_chat,
    messages=[
        {"role": "system", "content": SEARCH_INTENT_PROMPT.format(user_query=user_query)},
    ]
)

search_query = response.choices[0].message.content

print(search_query)


You are asked to read the user query, understand it, and then turn it into a search query.
The search query should be sent to the search engine to find the relevant information.
the search query should be similar to what a user would type into a search engine, and similar in meaning to the provided user query.
the user query is: 'am i covered for eye exams'
return only the search query text. 
"does insurance cover eye exams"


In [12]:
 # Retrieve the selected fields from the search index related to the question.
from azure.search.documents.models import (
    QueryType,
    QueryCaptionType,
    QueryAnswerType
)


vector_query = VectorizableTextQuery(text=user_query, k_nearest_neighbors=1, fields="vector", exhaustive=True)


results = search_client.search(  
    search_text=user_query,
    vector_queries=[vector_query],
    select=["parent_id", "chunk_id", "chunk", "title"],
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name='my-semantic-config',
    query_caption=QueryCaptionType.EXTRACTIVE,
    query_answer=QueryAnswerType.EXTRACTIVE,
    top=sources_to_include
)



semantic_answers = results.get_answers()
semantic_response = ""
if semantic_answers:
    for answer in semantic_answers:
        if answer.highlights:
            semantic_response = answer.highlights
            # print(f"Semantic Answer: {answer.highlights}")
        else:
            semantic_response = answer.text
            # print(f"Semantic Answer: {answer.text}")
        print(f"Semantic Answer: {semantic_response}")
        print(f"Semantic Answer Score: {answer.score}\n")

In [13]:
list_of_sources = [f'\nSOURCE NO {index + 1}. {result["title"]}({get_page_number(result["chunk_id"])}): {result["chunk"]} )' for index, result in enumerate(results)]

In [14]:
print(list_of_sources)

['\nSOURCE NO 1. Benefit_Options.pdf(1): care services, as well as prescription drug coverage. With \n\nNorthwind Standard, you can choose from a variety of in-network providers, including primary care \n\nphysicians, specialists, hospitals, and pharmacies. This plan does not offer coverage for emergency \n\nservices, mental health and substance abuse coverage, or out-of-network services. \n\nComparison of Plans  \nBoth plans offer coverage for routine physicals, well-child visits, immunizations, and other preventive \n\ncare services. The plans also cover preventive care services such as mammograms, colonoscopies, and \n\nother cancer screenings.  \n\nNorthwind Health Plus offers more comprehensive coverage than Northwind Standard. This plan offers \n\ncoverage for emergency services, both in-network and out-of-network, as well as mental health and \n\nsubstance abuse coverage. Northwind Standard does not offer coverage for emergency services, mental \n\nhealth and substance abuse cov

In [15]:
joined_sources = "\n".join(list_of_sources)

In [16]:
print(joined_sources)


SOURCE NO 1. Benefit_Options.pdf(1): care services, as well as prescription drug coverage. With 

Northwind Standard, you can choose from a variety of in-network providers, including primary care 

physicians, specialists, hospitals, and pharmacies. This plan does not offer coverage for emergency 

services, mental health and substance abuse coverage, or out-of-network services. 

Comparison of Plans  
Both plans offer coverage for routine physicals, well-child visits, immunizations, and other preventive 

care services. The plans also cover preventive care services such as mammograms, colonoscopies, and 

other cancer screenings.  

Northwind Health Plus offers more comprehensive coverage than Northwind Standard. This plan offers 

coverage for emergency services, both in-network and out-of-network, as well as mental health and 

substance abuse coverage. Northwind Standard does not offer coverage for emergency services, mental 

health and substance abuse coverage, or out-of-network

In [17]:
GROUNDED_PROMPT.format(sources=joined_sources, semantic_response=semantic_response)
print(GROUNDED_PROMPT.format(sources=joined_sources, semantic_response=semantic_response))


You are a friendly assistant that answer questions from company's HR policy and information documents.
Answer the query using only the sources provided below in a friendly and concise bulleted manner.
Answer ONLY with the facts listed in the list of sources below.
If there isn't enough information below, say you don't know.
Do not generate answers that don't use the sources below.
always provide the source of the information as a citation of source number in the list of sources.
Sources:

SOURCE NO 1. Benefit_Options.pdf(1): care services, as well as prescription drug coverage. With 

Northwind Standard, you can choose from a variety of in-network providers, including primary care 

physicians, specialists, hospitals, and pharmacies. This plan does not offer coverage for emergency 

services, mental health and substance abuse coverage, or out-of-network services. 

Comparison of Plans  
Both plans offer coverage for routine physicals, well-child visits, immunizations, and other preven

In [19]:
response = client.chat.completions.create(
    model=deployment_name_chat,
    messages=[
        {"role": "system", "content": GROUNDED_PROMPT.format(sources=joined_sources, semantic_response=semantic_response)},
        {"role": "user", "content": user_query}
    ]
)

In [20]:
print(user_query)
print(search_query)

am i covered for eye exams
"does insurance cover eye exams"


In [21]:


print(response.choices[0].message.content)


- Yes, both Northwind Health Plus and Northwind Standard offer coverage for routine eye exams. [Source No. 1; Source No. 3; Source No. 4]
